# Cleaning Cancer Registry
goal: 
Cancer Date
Cancer Site
Cancer Type

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# cancer registry
cancer = pd.read_stata(r"S:\LLC_0002\data\stata_w_labs\nhsd_CANCER_v0002_20230302.dta", preserve_dtypes = True)
cancer.columns

In [ ]:
# date
cancer = cancer.rename(columns = {"cancer_anniversary_day": "day", "cancer_anniversary_month": "month", "cancer_anniversary_year": "year"})
cancer['year'] = cancer['year'].apply(
    lambda x: np.nan if pd.isna(x) else (2000 + int(x) if x <25 else 1900 + int(x))
)

cancer['cancer_date'] = pd.to_datetime(cancer[['year', 'month', 'day']])


In [ ]:
# inspect missingness, 
cancer[cancer['cancer_date'].isna()]
# 103 missing

In [ ]:
# cancer type - combining type + behaviour to create icd-o-3 codes
cancer['cancer_type'] =  cancer['cancer_type'].astype("Int64") # 69 missing
cancer['cancer_behaviour'] = cancer['cancer_behaviour'].astype("Int64") # 141 missing
# Option to fill missing type as 3? Decide to not fill.
# cancer['cancer_behaviour'] = cancer['cancer_behaviour'].fillna(3).astype("Int64") # 141 missing


In [ ]:
# derive icd_o_3
cancer['icd_o_3'] = cancer['cancer_type'].astype(str) + "/" + cancer['cancer_behaviour'].astype(str)

# replace icd_o_3 as Missing if cancer_type is missing.
cancer.loc[cancer['cancer_type'].isna(), "icd_o_3"] = "Missing"

In [ ]:
# minimisation: 
# keep id, extract date, icd_o_3, to map with excel sheet (to-be-imported)
selected_vars = ["llc_0002_stud_id", "icd_o_3", "cancer_site", "derived_can_dobym", "cancer_date"]
df_cancer_minimised = cancer[selected_vars] 
# df_cancer_minimised.to_csv("cancer_registry_raw.csv")

In [ ]:
df_cancer_minimised

In [ ]:
# read icd-o-3 codes (access from seer.cancer.gov/icd-o-3)
df_icdo3 = pd.read_excel("sitetype_icdo3_d20220429.xlsx")


In [ ]:
# drop duplicates
df_icdo3 = df_icdo3[['Histology/Behavior', 'Histology/Behavior Description']].drop_duplicates()
df_icdo3 = df_icdo3.drop_duplicates(subset=['Histology/Behavior'], keep = 'first')

In [ ]:
# merge
merged_cancer = df_cancer_minimised.merge(df_icdo3[['Histology/Behavior', 'Histology/Behavior Description']], how = 'left', left_on = "icd_o_3", right_on = 'Histology/Behavior')
merged_cancer['flag'] = merged_cancer['Histology/Behavior Description'].isna().astype(int)


In [ ]:
# inspect missing = 1183
merged_cancer[merged_cancer['flag'] == 1]


In [ ]:
# use second list, from kegg
import json 

with open("br08420.json", 'r') as f:
    json_data = json.load(f)
      

In [ ]:
# extracting relevant descriptions

def extract_codes(data):
    mapping = {}
    if "children" in data:
        for child in data['children']:
            if "children" in child:
                mapping.update(extract_codes(child))
            else:
                name_split = child["name"].split("  ",1)
                if len(name_split) == 2:
                    code, description = name_split
                    mapping[code.strip()] = description.strip()
    return mapping

histology_mapping = extract_codes(json_data)



In [ ]:
df_mapping = pd.DataFrame(list(histology_mapping.items()), columns = ["Histology/Behaviour", "Description"])

In [ ]:
merged_cancer_2 = merged_cancer.merge(df_mapping[['Histology/Behaviour', 'Description']], how = 'left', left_on = "icd_o_3", right_on = 'Histology/Behaviour')

In [ ]:
# inspect
merged_cancer_2['flag'] = merged_cancer_2['Description'].isna().astype(int)
merged_cancer_2[merged_cancer_2['flag'] == 1] 
# 943 cases not included from this list




In [ ]:
# combining results from 2 lists

merged_cancer_2['Final_description'] =merged_cancer_2['Histology/Behavior Description'].combine_first(merged_cancer_2['Description'])

In [ ]:
# inspect: 765 cases with no code
merged_cancer_2['flag'] = merged_cancer_2['Final_description'].isna().astype(int)
merged_cancer_2[merged_cancer_2['flag'] == 1] 


In [ ]:
# next best information: just consider first 4 digits, ignore /(int)

# extract from hierarchy

def extract_hierarchy(data):
    hierarchy = {}
    if "children" in data:
        for child in data['children']:
            if "children" in child:
                sub_hierarchy = extract_hierarchy(child)
                hierarchy.update(sub_hierarchy)
            else:
                name_split = child["name"].split("  ", 1)
                if len(name_split) == 2:
                    code, description = name_split
                    parent_name = data["name"]
                    if parent_name not in hierarchy:
                        hierarchy[parent_name] = []
                    hierarchy[parent_name].append((code.strip(), description.strip()))
    return hierarchy


def extract_parent_mapping(data, parent_name = None):
    mapping = {}
    if "children" in data:
        for child in data['children']:
            if "children" in child:
                mapping.update(extract_parent_mapping(child, parent_name = child['name']))
            else:
                name_split = child["name"].split("  ", 1)
                if len(name_split) == 2:
                    code, _ = name_split
                    mapping[code.strip()] = parent_name
    return mapping


In [ ]:
hierarchy_dict = extract_hierarchy(json_data)
parent_mapping = extract_parent_mapping(json_data)

In [ ]:
# create df using prefix only
parent_map = pd.DataFrame(list(parent_mapping.items()), columns = ["child", "parent"])
parent_map['child_prefix'] = parent_map['child'].str[:4]
parent_map

In [ ]:
# merge with mapping
mapping = df_mapping.merge(parent_map, how = 'left', left_on = "Histology/Behaviour", right_on = 'child')

In [ ]:
# minimise
mapping = mapping[["Histology/Behaviour", "Description", "parent"]]

In [ ]:
# extract prefix
mapping["Prefix"] = mapping["Histology/Behaviour"].str[:4]

# keep unique (839 unique prefix)
mapping = mapping.drop_duplicates(subset=['Prefix'], keep = 'first')


In [ ]:
# get prefix
merged_cancer_2["Prefix"] = merged_cancer_2["icd_o_3"].str[:4]


In [ ]:
# merge
merged_cancer_3 = merged_cancer_2.merge(mapping[['Prefix', 'parent']], how = 'left', on = 'Prefix')

In [ ]:
# combine
merged_cancer_3['Final_final_description'] = merged_cancer_3['Final_description'].combine_first(merged_cancer_3['parent'])

In [ ]:
# inspect
merged_cancer_3['flag'] = merged_cancer_3['Final_final_description'].isna().astype(int)
merged_cancer_3[merged_cancer_3['flag'] == 1] # 180 remain missing.


In [ ]:
merged_cancer_3[merged_cancer_3['flag'] == 1]['icd_o_3'].value_counts()
# 69 have missing code, the rest (111) have invalid codes, which could be manually checked. 



In [ ]:
# recode the rest of 180 to cancer not otherwise specified
merged_cancer_3["Final_final_description"] = merged_cancer_3["Final_final_description"].fillna("Cancer NOS")

# minimisation - choice to keep "Final_description" without the NOS. 
merged_cancer_final = merged_cancer_3[["llc_0002_stud_id", "cancer_site", "cancer_date","derived_can_dobym", "icd_o_3","Final_final_description", "parent"]]
merged_cancer_final.rename(columns={"Final_final_description": "description"}, inplace = True)

In [ ]:
merged_cancer_final['parent'].value_counts(dropna = False) #212 NAs

In [ ]:
merged_cancer_final

In [ ]:
# read icd-o-3 site (access from seer.cancer.gov/icd-o-3)
df_icdo3 = pd.read_excel("sitetype_icdo3_d20220429.xlsx")
# drop dup
df_site =  df_icdo3[['Site recode','Site Description']]
df_site = df_site.drop_duplicates(subset=['Site recode', 'Site Description'], keep = 'first')
df_exploded = df_site.assign(**{"Site recode": df_site["Site recode"].str.split(",")}).explode("Site recode")

In [ ]:
# function to check range matches

def match_cancer_sites(cancer_sites_df, site_recode_df):
    def site_in_range(cancer_site, recode):
        if "-" in recode:
            start, end = recode.split("-")
            return start <= cancer_site <= end
        return cancer_site == recode
    
    match_dict = {}
    
    for _, row in site_recode_df.iterrows():
        recode = row["Site recode"]
        description = row["Site Description"]
        
        for site in cancer_sites_df["cancer_site"]:
            if site_in_range(site, recode):
                if site in match_dict:
                    match_dict[site].add(description)
                else:
                    match_dict[site] = {description}
    
    # read matches
    cancer_sites_df["Site Description"] = cancer_sites_df['cancer_site'].map(
        lambda x: "; ".join(match_dict[x]) if x in match_dict else None
    )
        
    return cancer_sites_df
        


In [ ]:
df_sites_updated = match_cancer_sites(merged_cancer_final, df_exploded)
# there are 179 site codes, and 7,744 rows with unknown or missing site description, but this could possibly be recoded from description/parent.
# I have recoded them to Missing for now


In [ ]:
df_sites_updated["Site Description"] =df_sites_updated["Site Description"].fillna("Missing or Unknown Code")


In [ ]:
df_sites_updated

In [ ]:
df_sites_updated.to_csv("cancer_registry_labelled.csv") 